In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')

from config import SPE1_PICKLE_ROOT, CELL_IDS, DICT_CELL_TYPE
from spikeparam.patch.fit import Spike

plt.rcParams['font.family'] = 'Helvetica Neue'

SPE1_PKL = SPE1_PICKLE_ROOT
PVC6_PKL = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles'

# Colours
COL_PC   = '#4878D0'   # spe-1 PC
COL_IN   = '#EE854A'   # spe-1 IN
COL_PVC6 = '#FF44CC'   # pvc-6 (both cells)

In [ ]:
FORCE_RERUN = False   # set True to refit pvc-6 and overwrite R² caches

## pvc-6 R² — refit from all_data pickles and cache

In [ ]:
def get_pvc6_r2(all_data_pkl, cache_pkl, fs=200000, force=False):
    """Fit Spike on pvc-6 all_data and cache r2_exp / r2_ramp."""
    if not force and os.path.exists(cache_pkl):
        with open(cache_pkl, 'rb') as f:
            d = pickle.load(f)
        # backfill old cache that used 'r_squared_exp' key
        if 'r_squared_exp' in d:
            d = {'r2_exp': d['r_squared_exp'], 'r2_ramp': d['r_squared_ramp']}
        return d

    with open(all_data_pkl, 'rb') as f:
        all_data = pickle.load(f)
    all_data_flat = [s for sweep in all_data for s in sweep]

    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=.008,
               pre_inflection_ms=0.5)
    sp.fit(all_data_flat, fs, n_jobs=-1, progress=tqdm)
    sp.gen_fit(ramp=True, exp=True)

    result = {
        'r2_exp':  np.asarray(sp.r_squared_exp,  dtype=float),
        'r2_ramp': np.asarray(sp.r_squared_ramp, dtype=float),
    }
    with open(cache_pkl, 'wb') as f:
        pickle.dump(result, f)
    print(f'Cached to {os.path.basename(cache_pkl)}')
    return result


pvc6_r2 = {
    'c1': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data.pkl'),
        os.path.join(PVC6_PKL, '_r2_c1.pkl'),
        force=FORCE_RERUN,
    ),
    'c2': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data2.pkl'),
        os.path.join(PVC6_PKL, '_r2_c2.pkl'),
        force=FORCE_RERUN,
    ),
}

for cid, d in pvc6_r2.items():
    print(f'pvc-6 {cid}: n={len(d["r2_exp"])}  '
          f'median r²_exp={np.nanmedian(d["r2_exp"]):.3f}  '
          f'median r²_ramp={np.nanmedian(d["r2_ramp"]):.3f}')

## spe-1 R² — load from spike_fit_pickles

In [ ]:
_fit_dir     = os.path.join(SPE1_PKL, 'spike_fit_pickles')
_cluster_dir = os.path.join(SPE1_PKL, 'cluster_pickles')

spe1_r2 = []   # list of dicts: cell_id, cell_type, r2_exp, r2_ramp

for _cid in CELL_IDS:
    # only include cells present in the main analysis (cluster pickles)
    _cluster_p = os.path.join(_cluster_dir, f'{_cid}_cluster_df.pkl')
    if not os.path.exists(_cluster_p):
        continue

    _fit_p = os.path.join(_fit_dir, f'{_cid}_spike_fit.pkl')
    if not os.path.exists(_fit_p):
        continue

    with open(_fit_p, 'rb') as _f:
        _sp = pickle.load(_f)

    if _sp.r_squared_exp is None:
        _sp.gen_fit(ramp=True, exp=True)

    _cnum  = int(_cid.replace('c', ''))
    _ctype = DICT_CELL_TYPE.get(_cnum, 'PC')
    spe1_r2.append({
        'cell_id':   _cid,
        'cell_type': _ctype,
        'r2_exp':    np.asarray(_sp.r_squared_exp,  dtype=float),
        'r2_ramp':   np.asarray(_sp.r_squared_ramp, dtype=float),
    })

print(f'Loaded {len(spe1_r2)} spe-1 cells (cluster-pickle gated)')
for d in spe1_r2:
    print(f"  {d['cell_id']} ({d['cell_type']}): n={len(d['r2_exp'])}  "
          f"med r²_exp={np.nanmedian(d['r2_exp']):.3f}  "
          f"med r²_ramp={np.nanmedian(d['r2_ramp']):.3f}")

## R² distributions — summary + per-cell

In [ ]:
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import pearsonr

matplotlib.rcParams['font.family']     = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Helvetica Neue', 'Helvetica', 'Arial']

_FS_AX  = 24
_FS_TK  = 18
_FS_SM  = 13   # small labels (per-cell x-ticks, scatter annotations)
_LW_SP  = 2.5
_ALPHA  = 0.80


def _pool(r2_key, ct=None):
    arrs = [d[r2_key][~np.isnan(d[r2_key])]
            for d in spe1_r2 if ct is None or d['cell_type'] == ct]
    return np.concatenate(arrs) if arrs else np.array([])


def _style(ax, xtick_sz=None):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp in ['bottom', 'left']:
        ax.spines[sp].set_linewidth(_LW_SP)
    ax.tick_params(axis='both', width=_LW_SP, labelsize=xtick_sz or _FS_TK)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')


def _boxes(ax, data, pos, cols, widths=0.62):
    bp = ax.boxplot(data, positions=pos, widths=widths, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2.5),
                    whiskerprops=dict(linewidth=_LW_SP * 0.9),
                    capprops=dict(linewidth=_LW_SP * 0.9),
                    showfliers=False)
    for p, c in zip(bp['boxes'], cols):
        p.set_facecolor(c); p.set_alpha(_ALPHA); p.set_linewidth(0)
    for w, c in zip(bp['whiskers'], [c for c in cols for _ in (0, 1)]):
        w.set_color(c); w.set_linewidth(_LW_SP * 0.9)
    for cp, c in zip(bp['caps'], [c for c in cols for _ in (0, 1)]):
        cp.set_color(c); cp.set_linewidth(_LW_SP * 0.9)
    return bp


def _set_r2_yax(ax):
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0', '0.25', '0.5', '0.75', '1'],
                       fontweight='bold', fontsize=_FS_TK)
    ax.axhline(0.5, color='#999999', ls='--', lw=1.5, alpha=0.5)


_pc_cells = [d for d in spe1_r2 if d['cell_type'] == 'PC']
_in_cells = [d for d in spe1_r2 if d['cell_type'] == 'IN']

fig = plt.figure(figsize=(24, 12))
gs = gridspec.GridSpec(2, 2, figure=fig,
                       hspace=0.56, wspace=0.36,
                       width_ratios=[1.1, 2.6])


# ── A: spe-1 summary — exp R² and ramp R² as 4 boxes ────────────────────────
ax_a = fig.add_subplot(gs[0, 0])

_boxes(ax_a,
       [_pool('r2_exp', 'PC'), _pool('r2_exp', 'IN'),
        _pool('r2_ramp', 'PC'), _pool('r2_ramp', 'IN')],
       [0, 1, 2.8, 3.8],
       [COL_PC, COL_IN, COL_PC, COL_IN],
       widths=0.72)

ax_a.axvline(1.9, color='#cccccc', lw=1.5, ls=':')
_set_r2_yax(ax_a)
ax_a.set_xticks([0, 1, 2.8, 3.8])
ax_a.set_xticklabels(['exp\nPC', 'exp\nIN', 'ramp\nPC', 'ramp\nIN'],
                     fontsize=_FS_TK - 2, fontweight='bold')
ax_a.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
ax_a.set_title('R²  –  spe-1 summary', fontsize=_FS_AX, fontweight='bold',
               pad=10, loc='left')
_style(ax_a, xtick_sz=_FS_TK - 2)
ax_a.tick_params(axis='y', labelsize=_FS_TK)
for lbl in ax_a.get_yticklabels():
    lbl.set_fontweight('bold')


# ── B: spe-1 per-cell exp R² ─────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])

b_data, b_cols, b_labs, b_pos = [], [], [], []
_x = 0
for d in _pc_cells:
    v = d['r2_exp']
    b_data.append(v[~np.isnan(v)]); b_cols.append(COL_PC)
    b_labs.append(d['cell_id']); b_pos.append(_x); _x += 1
sep_b = _x - 0.5; _x += 0.7
for d in _in_cells:
    v = d['r2_exp']
    b_data.append(v[~np.isnan(v)]); b_cols.append(COL_IN)
    b_labs.append(d['cell_id']); b_pos.append(_x); _x += 1

_boxes(ax_b, b_data, b_pos, b_cols)
ax_b.axvline(sep_b, color='#cccccc', lw=1.5, ls=':')
_set_r2_yax(ax_b)
ax_b.set_xticks(b_pos)
ax_b.set_xticklabels(b_labs, rotation=65, ha='right',
                     fontsize=_FS_SM, fontweight='bold')
ax_b.set_ylabel('Exp. decay  R²', fontsize=_FS_AX, fontweight='bold')
ax_b.set_title('Exp. decay R²  –  spe-1 per cell', fontsize=_FS_AX, fontweight='bold',
               pad=10, loc='left')
_style(ax_b, xtick_sz=_FS_SM)
ax_b.tick_params(axis='y', labelsize=_FS_TK)
for lbl in ax_b.get_yticklabels():
    lbl.set_fontweight('bold')


# ── C: scatter — exp R² vs ramp R² per-cell median ───────────────────────────
ax_c = fig.add_subplot(gs[1, 0])

for ct, col, cells in [('PC', COL_PC, _pc_cells), ('IN', COL_IN, _in_cells)]:
    xm = [np.nanmedian(d['r2_exp'])  for d in cells]
    ym = [np.nanmedian(d['r2_ramp']) for d in cells]
    ax_c.scatter(xm, ym, color=col, s=90, alpha=0.85, zorder=3, edgecolors='none',
                 label=f'spe-1 {ct}')

for cid, marker in [('c1', 'D'), ('c2', 's')]:
    ax_c.scatter(np.nanmedian(pvc6_r2[cid]['r2_exp']),
                 np.nanmedian(pvc6_r2[cid]['r2_ramp']),
                 color=COL_PVC6, s=200, marker=marker, alpha=0.9,
                 zorder=4, edgecolors='none', label=f'pvc-6 {cid}')

ax_c.plot([0, 1], [0, 1], color='#bbbbbb', ls='--', lw=1.5, zorder=1)
ax_c.set_xlim(-0.05, 1.05)
ax_c.set_ylim(-0.05, 1.05)
ax_c.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax_c.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax_c.set_xticklabels(['0', '0.25', '0.5', '0.75', '1'], fontweight='bold')
ax_c.set_yticklabels(['0', '0.25', '0.5', '0.75', '1'], fontweight='bold')
ax_c.set_xlabel('Exp. decay  R²', fontsize=_FS_AX, fontweight='bold')
ax_c.set_ylabel('Ramp  R²', fontsize=_FS_AX, fontweight='bold')
ax_c.set_title('Per-cell median R²', fontsize=_FS_AX, fontweight='bold',
               pad=10, loc='left')
_style(ax_c)

_xall = [np.nanmedian(d['r2_exp'])  for d in spe1_r2]
_yall = [np.nanmedian(d['r2_ramp']) for d in spe1_r2]
_r, _p = pearsonr(_xall, _yall)
ax_c.text(0.97, 0.05, f'r={_r:.2f}, {"p<0.001" if _p < 0.001 else f"p={_p:.3f}"}',
          transform=ax_c.transAxes, ha='right', va='bottom',
          fontsize=_FS_SM + 1, fontweight='bold', color='#444444')

ax_c.legend(frameon=False, prop={'size': _FS_SM + 1, 'weight': 'bold'},
            loc='upper left', handletextpad=0.4, borderpad=0.2)


# ── D: pvc-6 zoomed — exp + ramp for c1 and c2 ───────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])

d_data = [pvc6_r2['c1']['r2_exp'], pvc6_r2['c2']['r2_exp'],
          pvc6_r2['c1']['r2_ramp'], pvc6_r2['c2']['r2_ramp']]
d_data = [v[~np.isnan(v)] for v in d_data]

_boxes(ax_d, d_data, [0, 1, 2.8, 3.8], [COL_PVC6] * 4, widths=0.72)
ax_d.axvline(1.9, color='#cccccc', lw=1.5, ls=':')

_all_pvc = np.concatenate(d_data)
_ylo = max(0.0,  np.percentile(_all_pvc, 0.5)  - 0.04)
_yhi = min(1.02, np.percentile(_all_pvc, 99.5) + 0.04)
if _yhi - _ylo < 0.06:
    _mid = (_yhi + _ylo) / 2
    _ylo, _yhi = _mid - 0.04, _mid + 0.04
ax_d.set_ylim(_ylo, _yhi)
ax_d.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins=4, prune='both'))

ax_d.set_xticks([0, 1, 2.8, 3.8])
ax_d.set_xticklabels(['exp\nc1', 'exp\nc2', 'ramp\nc1', 'ramp\nc2'],
                     fontsize=_FS_TK - 2, fontweight='bold')
ax_d.set_ylabel('R²  (zoomed)', fontsize=_FS_AX, fontweight='bold')
ax_d.set_title('pvc-6 R²  (zoomed)', fontsize=_FS_AX, fontweight='bold',
               pad=10, loc='left')
_style(ax_d, xtick_sz=_FS_TK - 2)
ax_d.tick_params(axis='y', labelsize=_FS_TK)
for lbl in ax_d.get_yticklabels():
    lbl.set_fontweight('bold')


# ── shared color legend ───────────────────────────────────────────────────────
fig.legend(
    handles=[mpatches.Patch(facecolor=COL_PC,   alpha=_ALPHA, label='spe-1  PC'),
             mpatches.Patch(facecolor=COL_IN,   alpha=_ALPHA, label='spe-1  IN'),
             mpatches.Patch(facecolor=COL_PVC6, alpha=_ALPHA, label='pvc-6')],
    loc='upper right', frameon=False,
    prop={'size': _FS_AX, 'weight': 'bold'},
    bbox_to_anchor=(0.995, 1.0))

plt.savefig('supp_r2_distributions.pdf', bbox_inches='tight', dpi=300)
plt.savefig('supp_r2_distributions.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')